# Asset Pricing

* Universidad de Chile - ``Aplicaciones Financieras con Machine Learning``

* Franco Mansilla Ibáñez (www.francomansilla.com)

In [8]:
!pip install riskfolio-lib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.0/241.0 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 11.6 MB/s eta 0:00:00
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.7.1
    Uninstalling matplotlib-3.7.1:
      Successfully uninstalled matplotlib-3.7.1


In [9]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import yfinance as yf

from scipy.linalg import sqrtm
import scipy.stats as st
import statsmodels.api as sm
import riskfolio as rp
from statsmodels.stats.correlation_tools import cov_nearest

import cvxpy as cp

import warnings
warnings.filterwarnings('ignore')

In [35]:
assets = ['JCI', 'TGT', 'CMCSA',
          'CPB', 'MO', 'APA',
          'AAPL', 'JPM', 'MSFT']  # qué empresas son?
assets.sort()

start = '2022-01-01'
end = '2024-06-30'

# Tickers of factors
factors = ['MTUM', 'QUAL', 'VLUE', 'SIZE', 'USMV']
factors.sort()

tickers = assets + factors
tickers.sort()

# Downloading data
data = yf.download(tickers, start = start, end = end)
data = data.loc[:,('Adj Close', slice(None))]
data.columns = tickers
data = data.dropna()


[*********************100%***********************]  14 of 14 completed


In [36]:
# Factor returns
X = data[factors].pct_change().dropna()
# Assets returns
Y = data[assets].pct_change().dropna()

## 1.1. Stepwise Regression

In [37]:
y = Y['MSFT']
X1 = sm.add_constant(X)

ols = sm.OLS(y, X1).fit()
print(ols.summary())
betas = ols.params

                            OLS Regression Results                            
Dep. Variable:                   MSFT   R-squared:                       0.680
Model:                            OLS   Adj. R-squared:                  0.677
Method:                 Least Squares   F-statistic:                     262.3
Date:                Tue, 03 Sep 2024   Prob (F-statistic):          3.66e-150
Time:                        21:45:40   Log-Likelihood:                 1972.8
No. Observations:                 624   AIC:                            -3934.
Df Residuals:                     618   BIC:                            -3907.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -1.643e-05      0.000     -0.040      0.9

In [38]:
# La lista de factores que mejor explican una variable
best_factors = rp.forward_regression(X,
                                     Y['MSFT'],
                                     criterion='pvalue',
                                     threshold=0.05,
                                     verbose=False)
print(best_factors)

['QUAL', 'VLUE', 'SIZE']


In [39]:
# Betas de las Caracteristicas para cada Activo Y
step = 'Forward'
betas = rp.loadings_matrix(X=X,
                       Y=Y,
                       stepwise=step,
                       criterion='pvalue')
betas

,const,MTUM,QUAL,SIZE,USMV,VLUE
AAPL,-0.000135,-0.152734,1.658582,-0.414870,0.000000,0.000000
APA,0.000835,0.933812,-1.300437,0.741695,-0.941758,1.612301
CMCSA,-0.000245,-0.246299,0.000000,0.000000,0.510891,0.705887
CPB,0.000354,0.000000,-1.020292,0.000000,1.828036,0.000000
JCI,-0.000160,0.000000,0.000000,1.009895,0.000000,0.000000
JPM,0.000532,0.000000,0.000000,0.000000,0.000000,0.948743
MO,0.000365,0.000000,-0.375456,-0.505550,0.757104,0.845132
MSFT,-0.000007,0.000000,1.936823,-0.322605,0.000000,-0.565081
TGT,-0.000477,-0.379341,0.000000,1.010196,0.617123,0.000000


Modelo Factorial de Microsoft

$$ R_{msft} = const + 1.936 \cdot Qual - 0.3226 \cdot Size - 0.5650 \cdot VLUE$$

In [40]:
# Expectativa de Retorno Factores (incluye constante)
prom_factor = sm.add_constant(X).mean().to_numpy().reshape(-1,1)
prom_factor

array([[1.00000000e+00],
       [2.41772310e-04],
       [3.87283086e-04],
       [1.37138263e-04],
       [1.76086751e-04],
       [6.48658397e-05]])

In [41]:
# Cálcula Expectativa de Retorno por Activo
ret_expectativa = betas.to_numpy() @ prom_factor
ret_expectativa

array([[ 4.13920047e-04],
       [ 5.97191663e-04],
       [-1.68718695e-04],
       [ 2.80753724e-04],
       [-2.14890120e-05],
       [ 5.93680435e-04],
       [ 3.38175934e-04],
       [ 6.62172840e-04],
       [-3.21072706e-04]])

In [42]:
# COV de Factores
cov_var_factores = sm.add_constant(X).cov().to_numpy()
cov_var_factores

array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
        0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 1.55525431e-04, 1.29525162e-04, 1.17234239e-04,
        8.24088326e-05, 1.06802251e-04],
       [0.00000000e+00, 1.29525162e-04, 1.47777475e-04, 1.35803771e-04,
        9.25463260e-05, 1.20679554e-04],
       [0.00000000e+00, 1.17234239e-04, 1.35803771e-04, 1.42910144e-04,
        9.13711504e-05, 1.27871197e-04],
       [0.00000000e+00, 8.24088326e-05, 9.25463260e-05, 9.13711504e-05,
        7.44792579e-05, 8.33568757e-05],
       [0.00000000e+00, 1.06802251e-04, 1.20679554e-04, 1.27871197e-04,
        8.33568757e-05, 1.28690056e-04]])

In [43]:
# Calculo de Errores
errores = Y - sm.add_constant(X) @ betas.T
errores

,AAPL,APA,CMCSA,CPB,JCI,JPM,MO,MSFT,TGT
Date,,,,,,,,,
2022-01-04 00:00:00+00:00,-0.009297,0.016586,-0.016967,0.013278,0.008143,0.021069,0.011557,-0.003723,0.006314
2022-01-05 00:00:00+00:00,0.000079,-0.021768,0.003762,0.009104,0.003946,-0.012015,-0.012287,-0.002232,-0.007430
2022-01-06 00:00:00+00:00,-0.017082,0.031390,0.007320,0.025140,0.016120,0.007809,0.013383,-0.007956,0.014146
2022-01-07 00:00:00+00:00,0.014003,-0.009230,-0.011571,0.015000,-0.010511,0.006255,0.005923,0.019560,-0.000067
2022-01-10 00:00:00+00:00,0.006692,-0.019723,0.011047,-0.000499,-0.006940,-0.000584,0.012573,0.008840,-0.007436
...,...,...,...,...,...,...,...,...,...
2024-06-24 00:00:00+00:00,0.009556,0.042101,-0.009934,0.002180,0.009331,0.006907,0.006795,0.007334,0.011569
2024-06-25 00:00:00+00:00,-0.003725,-0.006126,0.007130,-0.010713,-0.011234,0.001065,0.006749,-0.006558,-0.007214
2024-06-26 00:00:00+00:00,0.021625,-0.007014,0.000190,0.037637,-0.021166,0.000561,0.002551,0.003010,0.008640


La matriz  $\Sigma_E$  es crucial en modelos de factor porque representa las incertidumbres o riesgos específicos de cada activo que no están relacionados con los movimientos del mercado más amplio o factores modelados.

$$\Sigma_E = \text{diag}(\text{Cov}(\epsilon))$$

In [44]:
# Varianza de los errores
sigma_e = np.diag(np.diag(errores.cov().to_numpy())) # Assume is diagonal
sigma_e

array([[0.00011941, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.0005829 , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.00017536, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.00014391, 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.00017955,
        0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.0001222 , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.00012532, 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.00010552, 0.        ],


$\sigma_{FM}$ cómo se espera que varíen conjuntamente los activos en un portafolio, considerando tanto las influencias comunes (factores) como las características únicas de cada activo.

$$ \sigma_{FM} = \beta \Sigma_{F} \beta^T + \Sigma_{E} $$

*	 $\sigma_{FM}$  es la matriz de covarianza de los activos basada en factores.
*	 $\beta$ es la matriz de betas de los activos con respecto a los factores.
*	 $\Sigma_{F}$  es la matriz de covarianza de los factores.
*	 $\beta^T$ es la transpuesta de la matriz de betas.
*	 $\Sigma_{E}$  es la matriz diagonal que contiene las varianzas específicas de los activos que no son explicadas por los factores.

In [45]:
# Factor Based Asset covariance matrix
sigma_fm = betas.to_numpy() @ cov_var_factores @ betas.to_numpy().T + sigma_e
sigma_fm

array([[3.16502352e-04, 1.36948651e-04, 1.09866944e-04, 1.58850148e-05,
        1.49511880e-04, 1.24090606e-04, 5.02333505e-05, 2.05608884e-04,
        1.59089122e-04],
       [1.36948651e-04, 8.55799055e-04, 1.23899712e-04, 2.23197841e-05,
        1.60556778e-04, 1.58083053e-04, 7.61391404e-05, 1.19740034e-04,
        1.55237212e-04],
       [1.09866944e-04, 1.23899712e-04, 2.70605516e-04, 3.74108048e-05,
        1.09137922e-04, 1.01630870e-04, 5.61296911e-05, 9.93812436e-05,
        1.26399812e-04],
       [1.58850148e-05, 2.23197841e-05, 3.74108048e-05, 2.01412965e-04,
        2.87519317e-05, 2.77516471e-05, 3.50100227e-05, 9.92724458e-06,
        4.74958039e-05],
       [1.49511880e-04, 1.60556778e-04, 1.09137922e-04, 2.87519317e-05,
        3.25297216e-04, 1.22517262e-04, 5.45433568e-05, 1.46098145e-04,
        1.57829108e-04],
       [1.24090606e-04, 1.58083053e-04, 1.01630870e-04, 2.77516471e-05,
        1.22517262e-04, 2.38037304e-04, 5.87412042e-05, 1.13624012e-04,
        1.3

* Una matriz de covarianza debe ser siempre semidefinida positiva. Esto significa que no debería tener valores propios (eigenvalues) negativos, ya que estos implicarían varianzas negativas. Sin embargo, en la práctica, cuando estimamos matrices de covarianza a partir de datos reales o aplicamos operaciones matemáticas como las que involucran a los factores y betas, pueden surgir imprecisiones numéricas o estimaciones sesgadas que resultan en una matriz de covarianza que no es semidefinida positiva.

* Este método específico dentro de cov_nearest ajusta la matriz modificando los valores propios que sean negativos a cero (o un umbral mínimo pequeño)

* En otras palabras:  La matriz de covanriazan historica por construccion es semidefinida positiva. En cambio la matriz de covarianza de factores no tiene nada que garantice que sea semidefinida positiva

In [ ]:
sigma_fm = cov_nearest(sigma_fm, method='clipped') # delete negative eigenvalues
sigma_fm

In [53]:
n_obs, n_col = Y.shape

x = cp.Variable((n_col, 1))

risk = cp.quad_form(x, sigma_fm)

ret = ret_expectativa.T @ x

constraints = [cp.sum(x) == 1,
               x >= 0]

objective = cp.Minimize(risk)
problem = cp.Problem(objective, constraints)
problem.solve()

weights = pd.DataFrame(x.value, index=assets)
print(round(weights,6))

              0
AAPL   0.041354
APA    0.000177
CMCSA  0.088638
CPB    0.311346
JCI    0.042390
JPM    0.107380
MO     0.313385
MSFT   0.095331
TGT    0.000000


In [59]:
prom_activos = Y.mean().to_numpy().reshape(-1,1)
cov_var = Y.cov().to_numpy()

ret_portafolio = weights.T @ prom_activos

print('Rentabilidad Portafolio:', ret_portafolio[0])
print('Riesgo Portafolio:', (x.value.T @ cov_var @ x.value)**0.5)

Rentabilidad Portafolio: 0    0.000322
Name: 0, dtype: float64
Riesgo Portafolio: [[0.00930927]]
